In [1]:
# ==========================================
# 🏆 IEEE 69-Bus N-1 专属压铸机 (纯净版)
# 核心：纯 Pandapower 原生，无 Numba，无基态，只造 N-1 灾难！
# ==========================================
import sys, os

# 临时屏蔽 stderr
devnull = open(os.devnull, 'w')
original_stderr = sys.stderr
sys.stderr = devnull
import pandapower as pp
import numpy as np
import pandas as pd
from tqdm import tqdm
import copy
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. 手动搭建 IEEE 69 节点系统
# ==========================================
def create_standard_69_bus():
    net = pp.create_empty_network()

    # 0 是高压侧平衡节点，1-69 是实际分布节点
    for i in range(70):
        pp.create_bus(net, vn_kv=12.66, name=f"Bus {i}")

    pp.create_ext_grid(net, bus=0, vm_pu=1.0)

    line_data = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
        (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18),
        (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26),
        (2, 27), (27, 28), (28, 29), (29, 30), (30, 31), (31, 32), (32, 33), (33, 34),
        (7, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 42),
        (42, 43), (43, 44), (44, 45), (3, 46), (46, 47), (47, 48), (48, 49), (49, 50),
        (50, 51), (51, 52), (8, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58),
        (58, 59), (10, 60), (60, 61), (61, 62), (62, 63), (63, 64), (11, 65), (65, 66),
        (66, 67), (67, 68)
    ]

    for f, t in line_data:
        pp.create_line_from_parameters(net, from_bus=f, to_bus=t, length_km=1.0,
                                     r_ohm_per_km=0.12, x_ohm_per_km=0.06, 
                                     c_nf_per_km=0, max_i_ka=0.3)
        # 初始负荷占位
        pp.create_load(net, bus=t, p_mw=0.02, q_mvar=0.01)

    return net

# ==========================================
# 2. 获取基准导纳矩阵 (仅用于后续做减法)
# ==========================================
def get_base_physics_matrices(net):
    pp.runpp(net)
    Y_bus = net._ppc['internal']['Ybus'].todense()
    # 注意：我们这里不保存基态矩阵，只返回它供 N-1 切割使用
    return Y_bus

# ==========================================
# 3. N-1 物理矩阵无损剥离 (保证 70x70 维度)
# ==========================================
def get_cut_matrices(Y_base, f, t):
    Y_cut = Y_base.copy()
    y_line = -Y_base[f, t] 

    Y_cut[f, f] -= y_line
    Y_cut[t, t] -= y_line
    Y_cut[f, t] += y_line 
    Y_cut[t, f] += y_line 

    return np.real(Y_cut), np.imag(Y_cut)

# ==========================================
# 4. 识别物理死区 (BFS 算法)
# ==========================================
def get_dead_zone(net, cut_f, cut_t):
    adj = {i: [] for i in range(70)}
    for _, r in net.line.iterrows():
        u, v = int(r.from_bus), int(r.to_bus)
        if (u == cut_f and v == cut_t) or (v == cut_f and u == cut_t):
            continue
        adj[u].append(v)
        adj[v].append(u)

    alive = set([0])
    q = [0]
    while q:
        curr = q.pop(0)
        for nxt in adj[curr]:
            if nxt not in alive:
                alive.add(nxt)
                q.append(nxt)

    dead = [i for i in range(70) if i not in alive]
    return dead

# ==========================================
# 5. 专属 N-1 数据压铸流水线
# ==========================================
def generate_n1_dataset(net_base, Y_base, case_name, cut_edge, num_samples=1000):
    print(f"\n🔥 正在全速生产战役: {case_name} | 断开线路: {cut_edge} | 样本量: {num_samples} ...")

    net_case = copy.deepcopy(net_base)
    f, t = cut_edge

    # 1. 物理断开线路
    line_idx = net_case.line[(net_case.line.from_bus == f) & (net_case.line.to_bus == t)].index[0]
    net_case.line.at[line_idx, 'in_service'] = False

    # 2. 识别并停用死区节点，防止求解器报错
    dead_nodes = get_dead_zone(net_case, f, t)
    net_case.bus.loc[dead_nodes, 'in_service'] = False
    print(f"   🚫 成功切除并隔离 {len(dead_nodes)} 个物理死区节点")

    # 3. 剥离并保存 N-1 的专属物理矩阵
    G_cut, B_cut = get_cut_matrices(Y_base, f, t)
    np.save(f'G_69_{case_name}.npy', G_cut)
    np.save(f'B_69_{case_name}.npy', B_cut)

    dataset = []
    np.random.seed(42)

    for i in tqdm(range(num_samples)):
        # 随机负荷波动
        for load_idx in net_case.load.index:
            p_mw_base = 0.02
            noise = np.random.uniform(0.8, 1.2)
            net_case.load.at[load_idx, 'p_mw'] = p_mw_base * noise
            net_case.load.at[load_idx, 'q_mvar'] = net_case.load.at[load_idx, 'p_mw'] * 0.3

        try:
            pp.runpp(net_case, enforce_q_lims=False)

            # 使用 fillna(0.0) 完美处理死区节点的 NaN，并归一化功率
            v_pu = net_case.res_bus.vm_pu.fillna(0.0).values
            v_ang = net_case.res_bus.va_degree.fillna(0.0).values
            p_inj = net_case.res_bus.p_mw.fillna(0.0).values / 100.0
            q_inj = net_case.res_bus.q_mvar.fillna(0.0).values / 100.0

            # 🚨 直接生成 [P, Q, V, Theta] 交替排列，完美适配 R-PINN 训练代码
            row = np.zeros(280)
            row[0::4] = p_inj
            row[1::4] = q_inj
            row[2::4] = v_pu
            row[3::4] = v_ang

            dataset.append(row)
        except Exception as e:
            continue 

    df = pd.DataFrame(dataset)
    df.to_csv(f'data_69_{case_name}.csv', index=False)
    print(f"✅ {case_name} 数据集及物理矩阵已保存！有效样本: {len(df)}")

# ==========================================
# 主程序点火 (只跑 N-1，绝对不跑基态)
# ==========================================
if __name__ == "__main__":
    net_69 = create_standard_69_bus()

    print("🚀 提取基准 70x70 导纳矩阵 (仅用于后续 N-1 矩阵推导)...")
    Y_base = get_base_physics_matrices(net_69)

    # 专属定制：直接生产三个 N-1 Case (各 1000 条)
    generate_n1_dataset(net_69, Y_base, case_name="C1", cut_edge=(9, 10), num_samples=1000)
    generate_n1_dataset(net_69, Y_base, case_name="C2", cut_edge=(35, 36), num_samples=1000)
    generate_n1_dataset(net_69, Y_base, case_name="C3", cut_edge=(54, 55), num_samples=1000)

    print("\n🎉 纯净版 N-1 生产完毕！没基态，没 Numba，干脆利落！去喂给你的 R-PINN 吧！干！")

In [ ]:
# ==============================================================================
# 🏆 IEEE 69-Bus R-PINN Proposed Method - Final Platinum Edition
# 特点：兼容 Pandapower 新数据 | 动态切片 | 物理零位遮蔽 | 逐点对账单 | SCI出图
# 任务：300 轮基态双轨压榨 -> C1/C2/C3 Zero-shot 零样本双状态降维打击
# 修复：完美对齐 69节点专属惩罚项 (平方 Relu + 1e6 权重) | 强制弹图显示
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import os
import warnings

# 过滤讨厌的警告
warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 R-PINN 战神版点火成功 | 目标：69节点双状态(V&Theta)重构 | 核心: {device}")

# 69节点系统常量
NUM_NODES = 69
# 15% 观测点 (包含平衡节点 0)
obs_indices = [0, 8, 11, 20, 26, 34, 45, 52, 60]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵加载 (对齐 Pandapower 生成的矩阵)
# ------------------------------------------
try:
    G_raw = np.load('G_matrix_69.npy')
    B_raw = np.load('B_matrix_69.npy')
except:
    print("❌ 找不到基态物理矩阵 G_matrix_69.npy！请确保已运行数据生成脚本。")
    exit()

# ⚠️ 核心修复：Pandapower 生成的是 70x70，我们截取前 69 个节点，保留 0 号 Slack 节点
if G_raw.shape[0] >= 69:
    G_np, B_np = G_raw[:69, :69], B_raw[:69, :69]
else:
    G_np, B_np = G_raw, B_raw

G_tensor = torch.from_numpy(G_np).float().to(device)
B_tensor = torch.from_numpy(B_np).float().to(device)
print(f"🧬 基态物理矩阵加载成功，精确对齐节点数: {G_tensor.shape[0]}")

# ------------------------------------------
# 3. 独立封装的误差函数与 R-PINN 模型
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    theta_rad = theta_pred * (np.pi / 180.0)
    dT = theta_rad.unsqueeze(2) - theta_rad.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

def compute_loss_components(V_pred, theta_pred, P_real, Q_real, V_real, T_real, G, B, obs_idx):
    P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, G, B)
    P_loss = torch.mean((P_calc - P_real) ** 2)
    Q_loss = torch.mean((Q_calc - Q_real) ** 2)
    
    # 双状态约束：同步锁死 V 和 Theta 的 15% 观测点误差
    V_obs_pred, V_obs_real = V_pred[:, obs_idx], V_real[:, obs_idx]
    T_obs_pred, T_obs_real = theta_pred[:, obs_idx], T_real[:, obs_idx]
    obs_loss = torch.mean((V_obs_pred - V_obs_real) ** 2) + torch.mean((T_obs_pred - T_obs_real) ** 2)
    
    # 🌟 绝杀修复：惩罚项改为平方 Relu
    penalty_low = torch.nn.functional.relu(0.85 - V_pred)
    penalty_high = torch.nn.functional.relu(V_pred - 1.10)
    penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
    
    return P_loss, Q_loss, obs_loss, penalty

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=69):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        # 👑 【核心 ARS 机制】: 残差缩放
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        theta_pred = out[:, self.node_num:] * 0.005 + 0.0     
        
        # 🛡️ 硬锚定：平衡节点 V=1.0, Theta=0.0
        vm_pred_clone = vm_pred.clone(); theta_pred_clone = theta_pred.clone()
        vm_pred_clone[:, 0] = 1.0; theta_pred_clone[:, 0] = 0.0
        return vm_pred_clone, theta_pred_clone

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_loss, Q_loss, obs_loss, penalty = compute_loss_components(
            V_pred, theta_pred, P_real, Q_real, V_real, T_real, self.G, self.B, self.obs_idx)
        # 🌟 绝杀修复：惩罚项权重拉满 1e6
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_indices, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_indices:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 🚨 数据装载：严格继承 Pandapower 基态阵型
# ------------------------------------------
print("📂 正在进行基态数据装载与切片对齐...")
try:
    df = pd.read_csv('ieee69_dataset_50k.csv', dtype=np.float32)
    data_val = df.values 
except:
    print("❌ 找不到基态数据 ieee69_dataset_50k.csv！")
    exit()

# ⚠️ 核心切片：基于 Pandapower 生成的分块排列 [V, Theta, P, Q]，截取前 69 节点
V_raw, T_raw = data_val[:, 0:69], data_val[:, 70:139] 
P_raw, Q_raw = data_val[:, 140:209], data_val[:, 210:279]

X_input = np.concatenate([P_raw, Q_raw], axis=1) 
Y_label = np.concatenate([V_raw, T_raw], axis=1) 

P_phys = P_raw
Q_phys = Q_raw

train_size = 40000
X_tr_raw, X_te_raw = X_input[:train_size], X_input[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_phys[:train_size], Q_phys[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_tr_norm).float(), torch.from_numpy(Y_tr).float(),
    torch.from_numpy(P_tr_phys).float(), torch.from_numpy(Q_tr_phys).float()), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 极致基态训练 (300 轮 + 动态余弦退火)
# ------------------------------------------
model = PowerGridPINN(node_num=69).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮双状态深度重构训练 (Proposed Method)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # 传入真实的 V (by[:, :69]) 和 Theta (by[:, 69:]) 以供约束
        loss = loss_fn(vp, tp, bp, bq, by[:, :69], by[:, 69:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态全量双状态统计审计
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.from_numpy(X_te_norm).float().to(device)
    ty = torch.from_numpy(Y_te).float().to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
        
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all, t_true_all = ty[:, :69], ty[:, 69:]
    
    err_v_base = (v_pred_all - v_true_all).cpu().numpy()
    err_t_base = (t_pred_all - t_true_all).cpu().numpy()

mae_v_base = np.mean(np.abs(err_v_base))
rmse_v_base = np.sqrt(np.mean(err_v_base**2))
mae_t_base = np.mean(np.abs(err_t_base))
rmse_t_base = np.sqrt(np.mean(err_t_base**2))

print("\n" + "="*65)
print(f"🏆 [Base Case Validation] IEEE 69-Bus Proposed R-PINN")
print(f"🌍 Voltage (V) MAE  : {mae_v_base:.6e} p.u. | RMSE: {rmse_v_base:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t_base:.6e} deg  | RMSE: {rmse_t_base:.6e} deg")
print("="*65)

# ------------------------------------------
# 7. 辅助工具箱：SCI 级双子图升级版
# ------------------------------------------
def plot_academic_case_69_dual(v_true, v_pred, t_true, t_pred, mask, title, mae_v, rmse_v, mae_t, rmse_t, save_name):
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['axes.unicode_minus'] = False
    plt.style.use('seaborn-v0_8-paper')
    
    nodes = np.arange(69)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=600, sharex=True)
    
    dead_nodes = np.where(mask == 0)[0]
    
    # --- 图 (a): Voltage Magnitude ---
    if len(dead_nodes) > 0:
        ax1.fill_between(dead_nodes, -0.05, 1.15, color='gray', alpha=0.15, label='Topological Dead Zone')
        ax1.text(np.mean(dead_nodes), 0.5, 'ZERO-SHOT\nDEAD ZONE', color='darkred', 
                 fontsize=14, fontweight='bold', ha='center', va='center')
    else:
        ax1.axvspan(0, 68, color='gray', alpha=0.05, label='Blind Zone (No Sensor)')

    ax1.plot(nodes, v_true, 'k-', label='Ground Truth (Physics)', linewidth=2.0, zorder=3)
    ax1.plot(nodes, v_pred, 'r--', label='Proposed R-PINN', linewidth=1.5, zorder=4)
    
    live_obs = [idx for idx in obs_indices if mask[idx] == 1]
    if len(live_obs) > 0:
        ax1.scatter(live_obs, v_true[live_obs], color='blue', marker='*', s=200, label='PMU Sensors (15% Obs)', zorder=5)

    ax1.set_title(f"{title}: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
    ax1.set_ylim(-0.05, 1.15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)

    # --- 图 (b): Phase Angle ---
    if len(dead_nodes) > 0:
        ymin, ymax = min(t_true), max(t_true)
        y_margin = (ymax - ymin) * 0.1 if ymax != ymin else 1.0
        ax2.fill_between(dead_nodes, ymin - y_margin, ymax + y_margin, color='gray', alpha=0.15)
    else:
        ax2.axvspan(0, 68, color='gray', alpha=0.05)

    ax2.plot(nodes, t_true, 'k-', linewidth=2.0, zorder=3)
    ax2.plot(nodes, t_pred, 'b--', label='Proposed R-PINN (Theta)', linewidth=1.5, zorder=4)
    
    if len(live_obs) > 0:
        ax2.scatter(live_obs, t_true[live_obs], color='blue', marker='*', s=200, zorder=5)

    ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} deg | RMSE: {rmse_t:.4e} deg)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Bus Index", fontsize=12)
    ax2.set_ylabel("Phase Angle (degree)", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight', dpi=600)
    # ⚠️ 核心修复：强制显示图片，并且移除 plt.close() 确保画布不被杀掉
    plt.show()

# ------------------------------------------
# 8. 🌪️ Zero-Shot (零样本) N-1 双状态降维打击推演
# ------------------------------------------
print("\n" + "⚔️" * 15 + " 启动 N-1 Zero-shot 突发对账 " + "⚔️" * 15)

test_cases_n1 = [
    {"n": "C1 (Cut 9-10)", "csv": "data_69_C1.csv"},
    {"n": "C2 (Cut 35-36)", "csv": "data_69_C2.csv"},
    {"n": "C3 (Cut 54-55)", "csv": "data_69_C3.csv"}
]

model.eval()
with torch.no_grad():
    for case in test_cases_n1:
        try:
            df_c = pd.read_csv(case["csv"]).fillna(0.0)
            raw_c = df_c.values 
            
            # ⚠️ 核心切片：[P, Q, V, Theta] 提取前 69 节点
            P_c = raw_c[:, 0::4][:, :69]
            Q_c = raw_c[:, 1::4][:, :69]
            V_c = raw_c[:, 2::4][:, :69]
            T_c = raw_c[:, 3::4][:, :69] 
            
            X_c_raw = np.concatenate([P_c, Q_c], axis=1)
            X_c = torch.tensor(scaler.transform(X_c_raw), dtype=torch.float32).to(device)
            V_t = torch.tensor(V_c, dtype=torch.float32).to(device)
            T_t = torch.tensor(T_c, dtype=torch.float32).to(device)
            
            # 拓扑盲区物理识别：利用真实数据中的 0 电压构建天然掩码
            V_t_np = V_t[0].cpu().numpy()
            mask_np = np.where(V_t_np < 1e-3, 0.0, 1.0)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).to(device)
            
            # 零样本推理！同时拿到 V 和 Theta
            V_raw, T_raw = model(apply_blind_zone(X_c, obs_indices, PHYS_ZERO))
            
            V_final = V_raw * mask_t # 死区物理隔离抹零
            T_final = T_raw * mask_t 
            
            err_v_c = (V_final - V_t * mask_t).cpu().numpy()
            mae_v_c = np.mean(np.abs(err_v_c))
            rmse_v_c = np.sqrt(np.mean(err_v_c**2))
            
            # 同步结算相角误差
            err_t_c = (T_final - T_t * mask_t).cpu().numpy()
            mae_t_c = np.mean(np.abs(err_t_c))
            rmse_t_c = np.sqrt(np.mean(err_t_c**2))
            
            print(f"\n🔬 诊断场景: {case['n']}")
            print(f"   • Zero-shot V MAE : {mae_v_c:.6e} p.u. | RMSE: {rmse_v_c:.6e} p.u.")
            print(f"   • Zero-shot T MAE : {mae_t_c:.6e} deg  | RMSE: {rmse_t_c:.6e} deg")
            dead = np.where(mask_np == 0)[0]
            print(f"   🚫 成功隔离 {len(dead)} 个物理死区节点")
            
            plot_title = f"Zero-shot Reconstruction Profile: {case['n']}"
            # ⚠️ 核心修复：更新图片命名，凸显 Proposed Platinum 版本
            save_filename = f"ieee69_zeroshot_{case['n'][:2]}_Proposed_Platinum.png"
            
            # 调用全新的双轨画图工具！
            plot_academic_case_69_dual(
                v_true=(V_t[0] * mask_t).cpu().numpy(),
                v_pred=V_final[0].cpu().numpy(),
                t_true=(T_t[0] * mask_t).cpu().numpy(),
                t_pred=T_final[0].cpu().numpy(),
                mask=mask_np,
                title=plot_title,
                mae_v=mae_v_c, rmse_v=rmse_v_c,
                mae_t=mae_t_c, rmse_t=rmse_t_c,
                save_name=save_filename
            )
            print(f"   ✅ 高清双轨科研图已保存: {save_filename}")
            
        except Exception as e:
            print(f"❌ 运行故障 ({case['n']}): {e}")

print("\n🎉 IEEE 69 节点双状态(V&Theta)完全对齐！跑完收工，出图！")

🚀 R-PINN 战神版点火成功 | 目标：69节点双状态(V&Theta)重构 | 核心: cuda
📡 锁定的 15% PMU 观测点: [0, 8, 11, 20, 26, 34, 45, 52, 60]
🧬 基态物理矩阵加载成功，精确对齐节点数: 69
📂 正在进行基态数据装载与切片对齐...
🔥 开始 300 轮双状态深度重构训练 (Proposed Method)...
Epoch 0   | Avg Loss: 7.6857e+03 | LR: 9.999729e-04


In [ ]:
# ==============================================================================
# [Sum-Loss 极限精度版] IEEE 69-Bus R-PINN (命名: model1)
# 核心改变：物理/观测 Loss 聚合切换为 SUM，惩罚项钉死 MEAN (消融对照组)
# 特点：兼容 Pandapower 新数据 | 双状态双轨对账 | 物理零位遮蔽 | SCI双子图
# 修复：ARS相角对齐 (*0.005) | 强制弹图显示
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import os
import warnings

# 过滤讨厌的警告
warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 消融实验舱就绪! 算力: {device} | 模式: Sum-Loss (Model 1)")

# 69节点系统常量
NUM_NODES = 69
# 15% 观测点 (包含平衡节点 0)
obs_indices = [0, 8, 11, 20, 26, 34, 45, 52, 60]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵加载 (对齐 Pandapower 格式)
# ------------------------------------------
try:
    G_raw = np.load('G_matrix_69.npy')
    B_raw = np.load('B_matrix_69.npy')
except:
    print("❌ 找不到基态物理矩阵 G_matrix_69.npy！请确保已运行数据生成脚本。")
    exit()

# ⚠️ 核心修复：Pandapower 生成的是 70x70，我们截取前 69 个节点，保留 0 号 Slack 节点
if G_raw.shape[0] >= 69:
    G_np, B_np = G_raw[:69, :69], B_raw[:69, :69]
else:
    G_np, B_np = G_raw, B_raw

G_tensor = torch.from_numpy(G_np).float().to(device)
B_tensor = torch.from_numpy(B_np).float().to(device)
print(f"🧬 基态物理矩阵加载成功，精确对齐节点数: {G_tensor.shape[0]}")

# ------------------------------------------
# 3. 独立封装的误差函数与 R-PINN 模型 (Sum-Loss版)
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    theta_rad = theta_pred * (np.pi / 180.0)
    dT = theta_rad.unsqueeze(2) - theta_rad.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

def compute_loss_components_sum(V_pred, theta_pred, P_real, Q_real, V_real, T_real, G, B, obs_idx):
    P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, G, B)
    
    # 👑【消融核心 1】：物理方程强制采用 SUM 聚合
    P_loss = torch.sum((P_calc - P_real) ** 2)
    Q_loss = torch.sum((Q_calc - Q_real) ** 2)
    
    # 👑【消融核心 2】：观测误差强制采用 SUM 聚合，且必须包含 V 和 Theta 双状态！
    V_obs_pred, V_obs_real = V_pred[:, obs_idx], V_real[:, obs_idx]
    T_obs_pred, T_obs_real = theta_pred[:, obs_idx], T_real[:, obs_idx]
    obs_loss = torch.sum((V_obs_pred - V_obs_real) ** 2) + torch.sum((T_obs_pred - T_obs_real) ** 2)
    
    # 🛡️【基线铁律】：惩罚项永远钉死 MEAN 聚合，且使用平方 ReLU
    penalty_low = torch.nn.functional.relu(0.85 - V_pred)
    penalty_high = torch.nn.functional.relu(V_pred - 1.10)
    penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
    
    return P_loss, Q_loss, obs_loss, penalty

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=69):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        # ⚠️ 修复点：严格对齐 ARS 相角缩放参数
        theta_pred = out[:, self.node_num:] * 0.005 + 0.0     
        
        vm_pred_clone = vm_pred.clone(); theta_pred_clone = theta_pred.clone()
        vm_pred_clone[:, 0] = 1.0; theta_pred_clone[:, 0] = 0.0
        return vm_pred_clone, theta_pred_clone

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_loss, Q_loss, obs_loss, penalty = compute_loss_components_sum(
            V_pred, theta_pred, P_real, Q_real, V_real, T_real, self.G, self.B, self.obs_idx)
        # 🛡️【基线铁律】：惩罚项权重拉满 1e6
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_indices, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_indices:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 🚨 数据装载：严格继承 Pandapower 基态阵型
# ------------------------------------------
print("📂 正在进行数据装载与切片对齐...")
try:
    df = pd.read_csv('ieee69_dataset_50k.csv', dtype=np.float32)
    data_val = df.values 
except:
    print("❌ 找不到基态数据 ieee69_dataset_50k.csv！")
    exit()

V_raw, T_raw = data_val[:, 0:69], data_val[:, 70:139] 
P_raw, Q_raw = data_val[:, 140:209], data_val[:, 210:279]

X_input = np.concatenate([P_raw, Q_raw], axis=1) 
Y_label = np.concatenate([V_raw, T_raw], axis=1) 

P_phys = P_raw
Q_phys = Q_raw

train_size = 40000
X_tr_raw, X_te_raw = X_input[:train_size], X_input[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_phys[:train_size], Q_phys[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_tr_norm).float(), torch.from_numpy(Y_tr).float(),
    torch.from_numpy(P_tr_phys).float(), torch.from_numpy(Q_tr_phys).float()), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 🚀 model1 深度训练 (Sum-Loss Mode + 余弦退火)
# ------------------------------------------
model1 = PowerGridPINN(node_num=69).to(device)
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 启动 model1 深度训练 (Sum-Loss Strategy)...")
for epoch in range(300):
    p_weight = 10 if epoch < 100 else 5000
    model1.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model1(mask_bx)
        
        # ⚠️ 传入 V 和 Theta 的 Ground Truth
        loss = loss_fn(vp, tp, bp, bq, by[:, :69], by[:, 69:], p_weight, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model1.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态全量统计审计 (双状态对齐)
# ------------------------------------------
model1.eval()
with torch.no_grad():
    tx = torch.from_numpy(X_te_norm).float().to(device)
    ty = torch.from_numpy(Y_te).float().to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
        
    v_pred_all, t_pred_all = model1(t_mask)
    v_true_all, t_true_all = ty[:, :69], ty[:, 69:]
    
    err_v_base = (v_pred_all - v_true_all).cpu().numpy()
    err_t_base = (t_pred_all - t_true_all).cpu().numpy()

mae_v_base = np.mean(np.abs(err_v_base))
rmse_v_base = np.sqrt(np.mean(err_v_base**2))
mae_t_base = np.mean(np.abs(err_t_base))
rmse_t_base = np.sqrt(np.mean(err_t_base**2))

print("\n" + "="*65)
print(f"🏆 [model1 最终对账单] 基态性能 (Sum-Loss)")
print(f"🌍 Voltage (V) MAE  : {mae_v_base:.6e} p.u. | RMSE: {rmse_v_base:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t_base:.6e} deg  | RMSE: {rmse_t_base:.6e} deg")
print("="*65)

# ------------------------------------------
# 7. 辅助工具箱：SCI 级双子图升级版
# ------------------------------------------
def plot_academic_case_69_dual(v_true, v_pred, t_true, t_pred, mask, title, mae_v, rmse_v, mae_t, rmse_t, save_name):
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['axes.unicode_minus'] = False
    plt.style.use('seaborn-v0_8-paper')
    
    nodes = np.arange(69)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=600, sharex=True)
    
    dead_nodes = np.where(mask == 0)[0]
    
    # --- 图 (a): Voltage Magnitude ---
    if len(dead_nodes) > 0:
        ax1.fill_between(dead_nodes, -0.05, 1.15, color='gray', alpha=0.15, label='Topological Dead Zone')
        ax1.text(np.mean(dead_nodes), 0.5, 'ZERO-SHOT\nDEAD ZONE', color='darkred', 
                 fontsize=14, fontweight='bold', ha='center', va='center')
    else:
        ax1.axvspan(0, 68, color='gray', alpha=0.05, label='Blind Zone (No Sensor)')

    ax1.plot(nodes, v_true, 'k-', label='Ground Truth (Physics)', linewidth=2.0, zorder=3)
    ax1.plot(nodes, v_pred, 'r--', label='Model1 (Sum-Loss)', linewidth=1.5, zorder=4)
    
    live_obs = [idx for idx in obs_indices if mask[idx] == 1]
    if len(live_obs) > 0:
        ax1.scatter(live_obs, v_true[live_obs], color='blue', marker='*', s=200, label='PMU Sensors (15% Obs)', zorder=5)

    ax1.set_title(f"{title}: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
    ax1.set_ylim(-0.05, 1.15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)

    # --- 图 (b): Phase Angle ---
    if len(dead_nodes) > 0:
        ymin, ymax = min(t_true), max(t_true)
        y_margin = (ymax - ymin) * 0.1 if ymax != ymin else 1.0
        ax2.fill_between(dead_nodes, ymin - y_margin, ymax + y_margin, color='gray', alpha=0.15)
    else:
        ax2.axvspan(0, 68, color='gray', alpha=0.05)

    ax2.plot(nodes, t_true, 'k-', linewidth=2.0, zorder=3)
    ax2.plot(nodes, t_pred, 'b--', label='Model1 (Sum-Loss) Theta', linewidth=1.5, zorder=4)
    
    if len(live_obs) > 0:
        ax2.scatter(live_obs, t_true[live_obs], color='blue', marker='*', s=200, zorder=5)

    ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} deg | RMSE: {rmse_t:.4e} deg)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Bus Index", fontsize=12)
    ax2.set_ylabel("Phase Angle (degree)", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight', dpi=600)
    # ⚠️ 修复点：添加 plt.show() 并在循环中显示，移除 plt.close() 防止图被抹除
    plt.show()

# ------------------------------------------
# 8. 🌪️ Zero-Shot (零样本) N-1 双状态降维打击推演
# ------------------------------------------
print("\n" + "⚔️" * 15 + " 启动 N-1 Zero-shot 突发对账 " + "⚔️" * 15)

test_cases_n1 = [
    {"n": "C1 (Cut 9-10)", "csv": "data_69_C1.csv"},
    {"n": "C2 (Cut 35-36)", "csv": "data_69_C2.csv"},
    {"n": "C3 (Cut 54-55)", "csv": "data_69_C3.csv"}
]

model1.eval()
with torch.no_grad():
    for case in test_cases_n1:
        try:
            df_c = pd.read_csv(case["csv"]).fillna(0.0)
            raw_c = df_c.values 
            
            # ⚠️ 切片提取 [P, Q, V, Theta]
            P_c = raw_c[:, 0::4][:, :69]
            Q_c = raw_c[:, 1::4][:, :69]
            V_c = raw_c[:, 2::4][:, :69]
            T_c = raw_c[:, 3::4][:, :69]
            
            X_c_raw = np.concatenate([P_c, Q_c], axis=1)
            X_c = torch.tensor(scaler.transform(X_c_raw), dtype=torch.float32).to(device)
            V_t = torch.tensor(V_c, dtype=torch.float32).to(device)
            T_t = torch.tensor(T_c, dtype=torch.float32).to(device)
            
            V_t_np = V_t[0].cpu().numpy()
            mask_np = np.where(V_t_np < 1e-3, 0.0, 1.0)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).to(device)
            
            V_raw, T_raw = model1(apply_blind_zone(X_c, obs_indices, PHYS_ZERO))
            V_final = V_raw * mask_t 
            T_final = T_raw * mask_t
            
            err_v_c = (V_final - V_t * mask_t).cpu().numpy()
            mae_v_c = np.mean(np.abs(err_v_c))
            rmse_v_c = np.sqrt(np.mean(err_v_c**2))
            
            err_t_c = (T_final - T_t * mask_t).cpu().numpy()
            mae_t_c = np.mean(np.abs(err_t_c))
            rmse_t_c = np.sqrt(np.mean(err_t_c**2))
            
            print(f"\n🔬 诊断场景: {case['n']}")
            print(f"   • Zero-shot V MAE : {mae_v_c:.6e} p.u. | RMSE: {rmse_v_c:.6e} p.u.")
            print(f"   • Zero-shot T MAE : {mae_t_c:.6e} deg  | RMSE: {rmse_t_c:.6e} deg")
            dead = np.where(mask_np == 0)[0]
            print(f"   🚫 成功隔离 {len(dead)} 个物理死区节点")
            
            plot_title = f"Sum-Loss Model Profile: {case['n']}"
            save_filename = f"ieee69_sumloss_{case['n'][:2]}_Dual.png"
            
            plot_academic_case_69_dual(
                v_true=(V_t[0] * mask_t).cpu().numpy(),
                v_pred=V_final[0].cpu().numpy(),
                t_true=(T_t[0] * mask_t).cpu().numpy(),
                t_pred=T_final[0].cpu().numpy(),
                mask=mask_np,
                title=plot_title,
                mae_v=mae_v_c, rmse_v=rmse_v_c,
                mae_t=mae_t_c, rmse_t=rmse_t_c,
                save_name=save_filename
            )
            print(f"   ✅ 高清双轨科研图已显示并保存: {save_filename}")
            
        except Exception as e:
            print(f"❌ 运行故障 ({case['n']}): {e}")

print("\n🎉 Model1 (Sum-Loss) 终极对照组跑完收工！")

In [ ]:
# ==============================================================================
# [消融对照版 - 绝对裸奔版] IEEE 69-Bus Vanilla PINN (无 ARS + Mean Loss)
# 核心消融：绝对移除 ARS 缩放！让网络完全盲猜！(单一变量)
# 公平底线：对齐双状态(V&Theta)约束、对齐 1e6 平方 Relu 惩罚项、对齐 SCI 双轨出图
# 特点：全面兼容 Pandapower 数据格式 | 动态切片 | 物理零位遮蔽 
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import os
import warnings

# 过滤讨厌的警告
warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Vanilla PINN (绝对裸奔版) 就绪! 算力: {device}")

NUM_NODES = 69
obs_indices = [0, 8, 11, 20, 26, 34, 45, 52, 60]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵加载 (对齐 Pandapower 格式)
# ------------------------------------------
try:
    G_raw = np.load('G_matrix_69.npy')
    B_raw = np.load('B_matrix_69.npy')
except:
    print("❌ 找不到基态物理矩阵 G_matrix_69.npy！请先运行造物主脚本。")
    exit()

# ⚠️ 核心修复：Pandapower 生成的是 70x70，截取前 69 个节点，保留 0 号 Slack 节点
if G_raw.shape[0] >= 69:
    G_np, B_np = G_raw[:69, :69], B_raw[:69, :69]
else:
    G_np, B_np = G_raw, B_raw

G_tensor = torch.from_numpy(G_np).float().to(device)
B_tensor = torch.from_numpy(B_np).float().to(device)
print(f"🧬 基态物理矩阵加载成功，精确对齐节点数: {G_tensor.shape[0]}")

# ------------------------------------------
# 3. 独立误差函数与 Vanilla PINN 模型
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    theta_rad = theta_pred * (np.pi / 180.0)
    dT = theta_rad.unsqueeze(2) - theta_rad.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

def compute_loss_components_vanilla(V_pred, theta_pred, P_real, Q_real, V_real, T_real, G, B, obs_idx):
    """Mean Loss 聚合：保持和 Proposed Method 完全一致的误差计算方式"""
    P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, G, B)
    P_loss = torch.mean((P_calc - P_real) ** 2)
    Q_loss = torch.mean((Q_calc - Q_real) ** 2)
    
    # 🌟 绝杀对齐 1：双状态 Mean-Loss 观测锚定
    V_obs_pred, V_obs_real = V_pred[:, obs_idx], V_real[:, obs_idx]
    T_obs_pred, T_obs_real = theta_pred[:, obs_idx], T_real[:, obs_idx]
    obs_loss = torch.mean((V_obs_pred - V_obs_real) ** 2) + torch.mean((T_obs_pred - T_obs_real) ** 2)
    
    # 🌟 绝杀对齐 2：钉死平方 Relu 惩罚项
    penalty_low = torch.nn.functional.relu(0.85 - V_pred)
    penalty_high = torch.nn.functional.relu(V_pred - 1.10)
    penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
    
    return P_loss, Q_loss, obs_loss, penalty

class VanillaPINN(nn.Module):
    def __init__(self, node_num=69):
        super(VanillaPINN, self).__init__()
        self.node_num = node_num
        self.layers = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.layers(x)
        # 👑 【绝对消融核心】：没有任何体面！没有 ARS！网络直接硬输出！
        vm_pred = out[:, :self.node_num].clone()
        theta_pred = out[:, self.node_num:].clone()
        
        # 🛡️ 硬锚定：平衡节点 V=1.0, Theta=0.0 (这是物理法则，不是ARS，必须保留)
        vm_pred[:, 0] = 1.0; theta_pred[:, 0] = 0.0
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_loss, Q_loss, obs_loss, penalty = compute_loss_components_vanilla(
            V_pred, theta_pred, P_real, Q_real, V_real, T_real, self.G, self.B, self.obs_idx)
        # 🌟 绝杀对齐 3：1e6 的重磅越界罚款
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_indices, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_indices:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (严格对齐 Pandapower 数据格式)
# ------------------------------------------
print("📂 正在进行数据装载与切片对齐...")
try:
    df = pd.read_csv('ieee69_dataset_50k.csv', dtype=np.float32)
    data_val = df.values 
except:
    print("❌ 找不到基态数据 ieee69_dataset_50k.csv！")
    exit()

V_raw, T_raw = data_val[:, 0:69], data_val[:, 70:139] 
P_raw, Q_raw = data_val[:, 140:209], data_val[:, 210:279]

X_input = np.concatenate([P_raw, Q_raw], axis=1) 
Y_label = np.concatenate([V_raw, T_raw], axis=1) 

P_phys = P_raw
Q_phys = Q_raw

train_size = 40000
X_tr_raw, X_te_raw = X_input[:train_size], X_input[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_phys[:train_size], Q_phys[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_tr_norm).float(), torch.from_numpy(Y_tr).float(),
    torch.from_numpy(P_tr_phys).float(), torch.from_numpy(Q_tr_phys).float()), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 极致基态训练
# ------------------------------------------
model_vanilla = VanillaPINN(node_num=69).to(device)

optimizer = torch.optim.Adam(model_vanilla.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 Vanilla PINN 深度训练 (绝对裸奔版 - 无 ARS)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model_vanilla.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model_vanilla(mask_bx)
        
        # ⚠️ 传入 V 和 Theta 的 Ground Truth
        loss = loss_fn(vp, tp, bp, bq, by[:, :69], by[:, 69:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_vanilla.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态全量双状态统计审计
# ------------------------------------------
model_vanilla.eval()
with torch.no_grad():
    tx = torch.from_numpy(X_te_norm).float().to(device)
    ty = torch.from_numpy(Y_te).float().to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
        
    v_pred_all, t_pred_all = model_vanilla(t_mask)
    v_true_all, t_true_all = ty[:, :69], ty[:, 69:]
    
    err_v_base = (v_pred_all - v_true_all).cpu().numpy()
    err_t_base = (t_pred_all - t_true_all).cpu().numpy()

mae_v_base = np.mean(np.abs(err_v_base))
rmse_v_base = np.sqrt(np.mean(err_v_base**2))
mae_t_base = np.mean(np.abs(err_t_base))
rmse_t_base = np.sqrt(np.mean(err_t_base**2))

print("\n" + "="*65)
print(f"🏆 [Vanilla PINN 对账单] 基态性能 (绝对裸奔版)")
print(f"🌍 Voltage (V) MAE  : {mae_v_base:.6e} p.u. | RMSE: {rmse_v_base:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t_base:.6e} deg  | RMSE: {rmse_t_base:.6e} deg")
print("="*65)

# ------------------------------------------
# 7. 辅助工具箱：SCI 级双子图
# ------------------------------------------
def plot_academic_case_69_dual(v_true, v_pred, t_true, t_pred, mask, title, mae_v, rmse_v, mae_t, rmse_t, save_name):
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['axes.unicode_minus'] = False
    plt.style.use('seaborn-v0_8-paper')
    
    nodes = np.arange(69)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=600, sharex=True)
    
    dead_nodes = np.where(mask == 0)[0]
    
    # --- 图 (a): Voltage Magnitude ---
    if len(dead_nodes) > 0:
        ax1.fill_between(dead_nodes, -0.05, 1.15, color='gray', alpha=0.15, label='Topological Dead Zone')
        ax1.text(np.mean(dead_nodes), 0.5, 'ZERO-SHOT\nDEAD ZONE', color='darkred', 
                 fontsize=14, fontweight='bold', ha='center', va='center')
    else:
        ax1.axvspan(0, 68, color='gray', alpha=0.05, label='Blind Zone (No Sensor)')

    ax1.plot(nodes, v_true, 'k-', label='Ground Truth (Physics)', linewidth=2.0, zorder=3)
    ax1.plot(nodes, v_pred, 'r--', label='Vanilla PINN (Naked)', linewidth=1.5, zorder=4)
    
    live_obs = [idx for idx in obs_indices if mask[idx] == 1]
    if len(live_obs) > 0:
        ax1.scatter(live_obs, v_true[live_obs], color='blue', marker='*', s=200, label='PMU Sensors (15% Obs)', zorder=5)

    ax1.set_title(f"{title}: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
    ax1.set_ylim(-0.05, 1.15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)

    # --- 图 (b): Phase Angle ---
    if len(dead_nodes) > 0:
        ymin, ymax = min(t_true), max(t_true)
        y_margin = (ymax - ymin) * 0.1 if ymax != ymin else 1.0
        ax2.fill_between(dead_nodes, ymin - y_margin, ymax + y_margin, color='gray', alpha=0.15)
    else:
        ax2.axvspan(0, 68, color='gray', alpha=0.05)

    ax2.plot(nodes, t_true, 'k-', linewidth=2.0, zorder=3)
    ax2.plot(nodes, t_pred, 'b--', label='Vanilla PINN Theta', linewidth=1.5, zorder=4)
    
    if len(live_obs) > 0:
        ax2.scatter(live_obs, t_true[live_obs], color='blue', marker='*', s=200, zorder=5)

    ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} deg | RMSE: {rmse_t:.4e} deg)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Bus Index", fontsize=12)
    ax2.set_ylabel("Phase Angle (degree)", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight', dpi=600)
    plt.close()

# ------------------------------------------
# 8. 🌪️ Zero-Shot (零样本) N-1 双状态对账
# ------------------------------------------
print("\n" + "⚔️" * 15 + " 启动 N-1 Zero-shot 突发对账 " + "⚔️" * 15)

test_cases_n1 = [
    {"n": "C1 (Cut 9-10)", "csv": "data_69_C1.csv"},
    {"n": "C2 (Cut 35-36)", "csv": "data_69_C2.csv"},
    {"n": "C3 (Cut 54-55)", "csv": "data_69_C3.csv"}
]

model_vanilla.eval()
with torch.no_grad():
    for case in test_cases_n1:
        try:
            df_c = pd.read_csv(case["csv"]).fillna(0.0)
            raw_c = df_c.values 
            
            # ⚠️ 核心修复：切片提取 [P, Q, V, Theta]
            P_c = raw_c[:, 0::4][:, :69]
            Q_c = raw_c[:, 1::4][:, :69]
            V_c = raw_c[:, 2::4][:, :69]
            T_c = raw_c[:, 3::4][:, :69]
            
            X_c_raw = np.concatenate([P_c, Q_c], axis=1)
            X_c = torch.tensor(scaler.transform(X_c_raw), dtype=torch.float32).to(device)
            V_t = torch.tensor(V_c, dtype=torch.float32).to(device)
            T_t = torch.tensor(T_c, dtype=torch.float32).to(device)
            
            V_t_np = V_t[0].cpu().numpy()
            mask_np = np.where(V_t_np < 1e-3, 0.0, 1.0)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).to(device)
            
            V_raw, T_raw = model_vanilla(apply_blind_zone(X_c, obs_indices, PHYS_ZERO))
            V_final = V_raw * mask_t 
            T_final = T_raw * mask_t 
            
            err_v_c = (V_final - V_t * mask_t).cpu().numpy()
            mae_v_c = np.mean(np.abs(err_v_c))
            rmse_v_c = np.sqrt(np.mean(err_v_c**2))
            
            err_t_c = (T_final - T_t * mask_t).cpu().numpy()
            mae_t_c = np.mean(np.abs(err_t_c))
            rmse_t_c = np.sqrt(np.mean(err_t_c**2))
            
            print(f"\n🔬 诊断场景: {case['n']}")
            print(f"   • Zero-shot V MAE : {mae_v_c:.6e} p.u. | RMSE: {rmse_v_c:.6e} p.u.")
            print(f"   • Zero-shot T MAE : {mae_t_c:.6e} deg  | RMSE: {rmse_t_c:.6e} deg")
            dead = np.where(mask_np == 0)[0]
            print(f"   🚫 成功隔离 {len(dead)} 个物理死区节点")
            
            plot_title = f"Vanilla PINN Profile: {case['n']}"
            save_filename = f"ieee69_vanilla_{case['n'][:2]}_Dual.png"
            
            plot_academic_case_69_dual(
                v_true=(V_t[0] * mask_t).cpu().numpy(),
                v_pred=V_final[0].cpu().numpy(),
                t_true=(T_t[0] * mask_t).cpu().numpy(),
                t_pred=T_final[0].cpu().numpy(),
                mask=mask_np,
                title=plot_title,
                mae_v=mae_v_c, rmse_v=rmse_v_c,
                mae_t=mae_t_c, rmse_t=rmse_t_c,
                save_name=save_filename
            )
            print(f"   ✅ 高清双轨科研图已保存: {save_filename}")
            
        except Exception as e:
            print(f"❌ 运行故障 ({case['n']}): {e}")

print("\n🎉 Vanilla PINN (绝对裸奔版) 跑完收工！去拿这组烂数据反衬你的创新点吧！干！")

In [ ]:
# ==============================================================================
# 🥊 [终极沙包对照组] IEEE 69-Bus Vanilla PINN (5k Few-Shot + 无 ARS)
# 核心使命：证明在 5000 条少样本条件下，没有 ARS 装甲的传统 PINN 会彻底崩盘！
# 绝对消融：完全移除 vm * 0.1 和 theta * 0.005，让网络硬抗物理非线性！
# 出图逻辑：强制弹图！拿这组崩坏的对比图，去神化你的 Proposed Platinum Edition！
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import os
import warnings

# 过滤讨厌的警告
warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🥊 Vanilla PINN (5k少样本裸奔版) 就绪! 准备迎接崩溃... 算力: {device}")

NUM_NODES = 69
obs_indices = [0, 8, 11, 20, 26, 34, 45, 52, 60]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵加载 (对齐 Pandapower 格式)
# ------------------------------------------
try:
    G_raw = np.load('G_matrix_69.npy')
    B_raw = np.load('B_matrix_69.npy')
except:
    print("❌ 找不到基态物理矩阵 G_matrix_69.npy！请先运行造物主脚本。")
    exit()

if G_raw.shape[0] >= 69:
    G_np, B_np = G_raw[:69, :69], B_raw[:69, :69]
else:
    G_np, B_np = G_raw, B_raw

G_tensor = torch.from_numpy(G_np).float().to(device)
B_tensor = torch.from_numpy(B_np).float().to(device)
print(f"🧬 物理矩阵加载成功。")

# ------------------------------------------
# 3. 独立误差函数与 Vanilla PINN 模型
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    theta_rad = theta_pred * (np.pi / 180.0)
    dT = theta_rad.unsqueeze(2) - theta_rad.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

def compute_loss_components_vanilla(V_pred, theta_pred, P_real, Q_real, V_real, T_real, G, B, obs_idx):
    """Mean Loss 聚合：保持和 Proposed Method 完全一致的误差计算方式"""
    P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, G, B)
    P_loss = torch.mean((P_calc - P_real) ** 2)
    Q_loss = torch.mean((Q_calc - Q_real) ** 2)
    
    # 🌟 绝杀对齐 1：双状态 Mean-Loss 观测锚定
    V_obs_pred, V_obs_real = V_pred[:, obs_idx], V_real[:, obs_idx]
    T_obs_pred, T_obs_real = theta_pred[:, obs_idx], T_real[:, obs_idx]
    obs_loss = torch.mean((V_obs_pred - V_obs_real) ** 2) + torch.mean((T_obs_pred - T_obs_real) ** 2)
    
    # 🌟 绝杀对齐 2：钉死平方 Relu 惩罚项
    penalty_low = torch.nn.functional.relu(0.85 - V_pred)
    penalty_high = torch.nn.functional.relu(V_pred - 1.10)
    penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
    
    return P_loss, Q_loss, obs_loss, penalty

class VanillaPINN(nn.Module):
    def __init__(self, node_num=69):
        super(VanillaPINN, self).__init__()
        self.node_num = node_num
        self.layers = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.layers(x)
        # 👑 【核心消融使命】：没有任何体面！没有 ARS 护甲！网络直接硬输出全量绝对值！
        vm_pred = out[:, :self.node_num].clone()
        theta_pred = out[:, self.node_num:].clone()
        
        # 🛡️ 硬锚定：平衡节点 V=1.0, Theta=0.0 (这是物理法则，不能删)
        vm_pred[:, 0] = 1.0; theta_pred[:, 0] = 0.0
        return vm_pred, theta_pred

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_loss, Q_loss, obs_loss, penalty = compute_loss_components_vanilla(
            V_pred, theta_pred, P_real, Q_real, V_real, T_real, self.G, self.B, self.obs_idx)
        # 🌟 绝杀对齐 3：1e6 的重磅越界罚款
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_indices, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_indices:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 数据装载 (⚠️ 强制执行 5000 极少量样本拦截！)
# ------------------------------------------
print("📂 正在装载数据并执行 5k 极少数样本拦截...")
try:
    df = pd.read_csv('ieee69_dataset_50k.csv', dtype=np.float32)
    data_val = df.values 
except:
    print("❌ 找不到数据 ieee69_dataset_50k.csv！")
    exit()

V_raw, T_raw = data_val[:, 0:69], data_val[:, 70:139] 
P_raw, Q_raw = data_val[:, 140:209], data_val[:, 210:279]

X_input = np.concatenate([P_raw, Q_raw], axis=1) 
Y_label = np.concatenate([V_raw, T_raw], axis=1) 

P_phys = P_raw
Q_phys = Q_raw

# 🥊 【致命一击】：无情砍掉 35000 条训练数据，只留 5000 条！
train_size = 5000  # <--- 就是这里！让 Vanilla PINN 在数据饥荒中颤抖吧！
print(f"⚠️ 警告：训练集已被强行阉割至 {train_size} 条 (Few-Shot 模式启动)！")

X_tr_raw, X_te_raw = X_input[:train_size], X_input[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_phys[:train_size], Q_phys[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_tr_norm).float(), torch.from_numpy(Y_tr).float(),
    torch.from_numpy(P_tr_phys).float(), torch.from_numpy(Q_tr_phys).float()), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 5k 少样本挣扎训练
# ------------------------------------------
model_vanilla = VanillaPINN(node_num=69).to(device)

optimizer = torch.optim.Adam(model_vanilla.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 Vanilla PINN 挣扎训练 (绝对裸奔版 - 无 ARS + 5k样本)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model_vanilla.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model_vanilla(mask_bx)
        
        loss = loss_fn(vp, tp, bp, bq, by[:, :69], by[:, 69:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_vanilla.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态全量统计 (见证崩溃)
# ------------------------------------------
model_vanilla.eval()
with torch.no_grad():
    tx = torch.from_numpy(X_te_norm).float().to(device)
    ty = torch.from_numpy(Y_te).float().to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
        
    v_pred_all, t_pred_all = model_vanilla(t_mask)
    v_true_all, t_true_all = ty[:, :69], ty[:, 69:]
    
    err_v_base = (v_pred_all - v_true_all).cpu().numpy()
    err_t_base = (t_pred_all - t_true_all).cpu().numpy()

mae_v_base = np.mean(np.abs(err_v_base))
rmse_v_base = np.sqrt(np.mean(err_v_base**2))
mae_t_base = np.mean(np.abs(err_t_base))
rmse_t_base = np.sqrt(np.mean(err_t_base**2))

print("\n" + "="*65)
print(f"☠️ [Vanilla PINN 对账单] 基态性能 (无 ARS + 5k样本)")
print(f"🌍 Voltage (V) MAE  : {mae_v_base:.6e} p.u. | RMSE: {rmse_v_base:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t_base:.6e} deg  | RMSE: {rmse_t_base:.6e} deg")
print("="*65)

# ------------------------------------------
# 7. 辅助工具箱：SCI 级双子图 (强制弹图版)
# ------------------------------------------
def plot_academic_case_69_dual(v_true, v_pred, t_true, t_pred, mask, title, mae_v, rmse_v, mae_t, rmse_t, save_name):
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['axes.unicode_minus'] = False
    plt.style.use('seaborn-v0_8-paper')
    
    nodes = np.arange(69)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=600, sharex=True)
    
    dead_nodes = np.where(mask == 0)[0]
    
    # --- 图 (a): Voltage Magnitude ---
    if len(dead_nodes) > 0:
        ax1.fill_between(dead_nodes, -0.05, 1.15, color='gray', alpha=0.15, label='Topological Dead Zone')
        ax1.text(np.mean(dead_nodes), 0.5, 'ZERO-SHOT\nDEAD ZONE', color='darkred', 
                 fontsize=14, fontweight='bold', ha='center', va='center')
    else:
        ax1.axvspan(0, 68, color='gray', alpha=0.05, label='Blind Zone (No Sensor)')

    ax1.plot(nodes, v_true, 'k-', label='Ground Truth (Physics)', linewidth=2.0, zorder=3)
    ax1.plot(nodes, v_pred, 'r--', label='Vanilla PINN (Naked 5k)', linewidth=1.5, zorder=4)
    
    live_obs = [idx for idx in obs_indices if mask[idx] == 1]
    if len(live_obs) > 0:
        ax1.scatter(live_obs, v_true[live_obs], color='blue', marker='*', s=200, label='PMU Sensors (15% Obs)', zorder=5)

    ax1.set_title(f"{title}: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
    ax1.set_ylim(-0.05, 1.15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)

    # --- 图 (b): Phase Angle ---
    if len(dead_nodes) > 0:
        ymin, ymax = min(t_true), max(t_true)
        y_margin = (ymax - ymin) * 0.1 if ymax != ymin else 1.0
        ax2.fill_between(dead_nodes, ymin - y_margin, ymax + y_margin, color='gray', alpha=0.15)
    else:
        ax2.axvspan(0, 68, color='gray', alpha=0.05)

    ax2.plot(nodes, t_true, 'k-', linewidth=2.0, zorder=3)
    ax2.plot(nodes, t_pred, 'b--', label='Vanilla PINN Theta', linewidth=1.5, zorder=4)
    
    if len(live_obs) > 0:
        ax2.scatter(live_obs, t_true[live_obs], color='blue', marker='*', s=200, zorder=5)

    ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} deg | RMSE: {rmse_t:.4e} deg)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Bus Index", fontsize=12)
    ax2.set_ylabel("Phase Angle (degree)", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight', dpi=600)
    # 🥊 核心出图逻辑修复：强行保留画布并展示，移除了害人的 plt.close()！
    plt.show()

# ------------------------------------------
# 8. 🌪️ Zero-Shot (零样本) N-1 双状态降维打击 (准备看笑话)
# ------------------------------------------
print("\n" + "⚔️" * 15 + " 启动 N-1 Zero-shot 突发对账 " + "⚔️" * 15)

test_cases_n1 = [
    {"n": "C1 (Cut 9-10)", "csv": "data_69_C1.csv"},
    {"n": "C2 (Cut 35-36)", "csv": "data_69_C2.csv"},
    {"n": "C3 (Cut 54-55)", "csv": "data_69_C3.csv"}
]

model_vanilla.eval()
with torch.no_grad():
    for case in test_cases_n1:
        try:
            df_c = pd.read_csv(case["csv"]).fillna(0.0)
            raw_c = df_c.values 
            
            P_c = raw_c[:, 0::4][:, :69]
            Q_c = raw_c[:, 1::4][:, :69]
            V_c = raw_c[:, 2::4][:, :69]
            T_c = raw_c[:, 3::4][:, :69]
            
            X_c_raw = np.concatenate([P_c, Q_c], axis=1)
            X_c = torch.tensor(scaler.transform(X_c_raw), dtype=torch.float32).to(device)
            V_t = torch.tensor(V_c, dtype=torch.float32).to(device)
            T_t = torch.tensor(T_c, dtype=torch.float32).to(device)
            
            V_t_np = V_t[0].cpu().numpy()
            mask_np = np.where(V_t_np < 1e-3, 0.0, 1.0)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).to(device)
            
            V_raw, T_raw = model_vanilla(apply_blind_zone(X_c, obs_indices, PHYS_ZERO))
            V_final = V_raw * mask_t 
            T_final = T_raw * mask_t 
            
            err_v_c = (V_final - V_t * mask_t).cpu().numpy()
            mae_v_c = np.mean(np.abs(err_v_c))
            rmse_v_c = np.sqrt(np.mean(err_v_c**2))
            
            err_t_c = (T_final - T_t * mask_t).cpu().numpy()
            mae_t_c = np.mean(np.abs(err_t_c))
            rmse_t_c = np.sqrt(np.mean(err_t_c**2))
            
            print(f"\n☠️ 见证崩溃场景: {case['n']}")
            print(f"   • Zero-shot V5k MAE : {mae_v_c:.6e} p.u. | RMSE: {rmse_v_c:.6e} p.u.")
            print(f"   • Zero-shot T5k MAE : {mae_t_c:.6e} deg  | RMSE: {rmse_t_c:.6e} deg")
            dead = np.where(mask_np == 0)[0]
            print(f"   🚫 成功隔离 {len(dead)} 个物理死区节点")
            
            plot_title = f"Vanilla PINN Failure Profile: {case['n']}"
            save_filename = f"ieee69_vanilla_5k_{case['n'][:2]}_Dual.png"
            
            plot_academic_case_69_dual(
                v_true=(V_t[0] * mask_t).cpu().numpy(),
                v_pred=V_final[0].cpu().numpy(),
                t_true=(T_t[0] * mask_t).cpu().numpy(),
                t_pred=T_final[0].cpu().numpy(),
                mask=mask_np,
                title=plot_title,
                mae_v=mae_v_c, rmse_v=rmse_v_c,
                mae_t=mae_t_c, rmse_t=rmse_t_c,
                save_name=save_filename
            )
            print(f"   ✅ 崩溃的对照图已显示并保存: {save_filename}")
            
        except Exception as e:
            print(f"❌ 运行故障 ({case['n']}): {e}")

print("\n🎉 Vanilla PINN 跑完收工！快去拿这组惨不忍睹的数据放到表格里，反衬你的创新点吧！干！")

In [ ]:
# ==============================================================================
# 🏆 IEEE 69-Bus R-PINN Proposed Method - Final Platinum Edition
# 特点：兼容 Pandapower 新数据 | 动态切片 | 物理零位遮蔽 | 逐点对账单 | SCI出图
# 任务：300 轮基态双轨压榨 -> C1/C2/C3 Zero-shot 零样本双状态降维打击
# 修复：完美对齐 69节点专属惩罚项 (平方 Relu + 1e6 权重) | ARS 相角 0.005 | 强制出图
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import os
import warnings

# 过滤讨厌的警告
warnings.filterwarnings("ignore")

# ------------------------------------------
# 1. 环境配置与全局种子锁定
# ------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 R-PINN 战神版点火成功 | 目标：69节点双状态(V&Theta)重构 | 核心: {device}")

# 69节点系统常量
NUM_NODES = 69
# 15% 观测点 (包含平衡节点 0)
obs_indices = [0, 8, 11, 20, 26, 34, 45, 52, 60]
print(f"📡 锁定的 15% PMU 观测点: {obs_indices}")

# ------------------------------------------
# 2. 物理矩阵加载 (对齐 Pandapower 生成的矩阵)
# ------------------------------------------
try:
    G_raw = np.load('G_matrix_69.npy')
    B_raw = np.load('B_matrix_69.npy')
except:
    print("❌ 找不到基态物理矩阵 G_matrix_69.npy！请确保已运行数据生成脚本。")
    exit()

# ⚠️ 核心修复：Pandapower 生成的是 70x70，我们截取前 69 个节点，保留 0 号 Slack 节点
if G_raw.shape[0] >= 69:
    G_np, B_np = G_raw[:69, :69], B_raw[:69, :69]
else:
    G_np, B_np = G_raw, B_raw

G_tensor = torch.from_numpy(G_np).float().to(device)
B_tensor = torch.from_numpy(B_np).float().to(device)
print(f"🧬 基态物理矩阵加载成功，精确对齐节点数: {G_tensor.shape[0]}")

# ------------------------------------------
# 3. 独立封装的误差函数与 R-PINN 模型
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    theta_rad = theta_pred * (np.pi / 180.0)
    dT = theta_rad.unsqueeze(2) - theta_rad.unsqueeze(1)
    p_term = G_t * torch.cos(dT) + B_t * torch.sin(dT)
    q_term = G_t * torch.sin(dT) - B_t * torch.cos(dT)
    return V_pred * torch.sum(V_pred.unsqueeze(1) * p_term, dim=2), V_pred * torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

def compute_loss_components(V_pred, theta_pred, P_real, Q_real, V_real, T_real, G, B, obs_idx):
    P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, G, B)
    P_loss = torch.mean((P_calc - P_real) ** 2)
    Q_loss = torch.mean((Q_calc - Q_real) ** 2)
    
    # 双状态约束：同步锁死 V 和 Theta 的 15% 观测点误差
    V_obs_pred, V_obs_real = V_pred[:, obs_idx], V_real[:, obs_idx]
    T_obs_pred, T_obs_real = theta_pred[:, obs_idx], T_real[:, obs_idx]
    obs_loss = torch.mean((V_obs_pred - V_obs_real) ** 2) + torch.mean((T_obs_pred - T_obs_real) ** 2)
    
    # 🌟 绝杀修复：惩罚项改为平方 Relu
    penalty_low = torch.nn.functional.relu(0.85 - V_pred)
    penalty_high = torch.nn.functional.relu(V_pred - 1.10)
    penalty = torch.mean(torch.pow(penalty_low, 2) + torch.pow(penalty_high, 2))
    
    return P_loss, Q_loss, obs_loss, penalty

class PowerGridPINN(nn.Module):
    def __init__(self, node_num=69):
        super(PowerGridPINN, self).__init__()
        self.node_num = node_num
        self.net = nn.Sequential(
            nn.Linear(node_num * 2, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, node_num * 2) 
        )
    def forward(self, x):
        out = self.net(x)
        # 👑 【核心 ARS 机制】: 残差缩放
        vm_pred = out[:, :self.node_num] * 0.1 + 1.0  
        # ⚠️ 修复：相角强制采用 * 0.005 + 0.0 缩放！
        theta_pred = out[:, self.node_num:] * 0.005 + 0.0     
        
        # 🛡️ 硬锚定：平衡节点 V=1.0, Theta=0.0
        vm_pred_clone = vm_pred.clone(); theta_pred_clone = theta_pred.clone()
        vm_pred_clone[:, 0] = 1.0; theta_pred_clone[:, 0] = 0.0
        return vm_pred_clone, theta_pred_clone

class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx

    def forward(self, V_pred, theta_pred, P_real, Q_real, V_real, T_real, p_weight, obs_weight=5e6):
        P_loss, Q_loss, obs_loss, penalty = compute_loss_components(
            V_pred, theta_pred, P_real, Q_real, V_real, T_real, self.G, self.B, self.obs_idx)
        # 🌟 绝杀修复：惩罚项权重拉满 1e6
        return p_weight * (P_loss + Q_loss) + obs_weight * obs_loss + 1e6 * penalty

def apply_blind_zone(batch_x, obs_indices, phys_zero):
    mask_bx = phys_zero.repeat(batch_x.shape[0], 1).to(device)
    for idx in obs_indices:
        mask_bx[:, idx] = batch_x[:, idx]                 
        mask_bx[:, idx+NUM_NODES] = batch_x[:, idx+NUM_NODES]  
    return mask_bx

# ------------------------------------------
# 4. 🚨 数据装载：严格继承 Pandapower 基态阵型
# ------------------------------------------
print("📂 正在进行基态数据装载与切片对齐...")
try:
    df = pd.read_csv('ieee69_dataset_50k.csv', dtype=np.float32)
    data_val = df.values 
except:
    print("❌ 找不到基态数据 ieee69_dataset_50k.csv！")
    exit()

# ⚠️ 核心切片：基于 Pandapower 生成的分块排列 [V, Theta, P, Q]，截取前 69 节点
V_raw, T_raw = data_val[:, 0:69], data_val[:, 70:139] 
P_raw, Q_raw = data_val[:, 140:209], data_val[:, 210:279]

X_input = np.concatenate([P_raw, Q_raw], axis=1) 
Y_label = np.concatenate([V_raw, T_raw], axis=1) 

P_phys = P_raw
Q_phys = Q_raw

train_size = 5000
X_tr_raw, X_te_raw = X_input[:train_size], X_input[train_size:]
Y_tr, Y_te = Y_label[:train_size], Y_label[train_size:]
P_tr_phys, Q_tr_phys = P_phys[:train_size], Q_phys[:train_size]

scaler = StandardScaler()
X_tr_norm = scaler.fit_transform(X_tr_raw)
X_te_norm = scaler.transform(X_te_raw)

mean_t = torch.tensor(scaler.mean_, device=device).float()
scale_t = torch.tensor(scaler.scale_, device=device).float()
PHYS_ZERO = (0.0 - mean_t) / scale_t

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_tr_norm).float(), torch.from_numpy(Y_tr).float(),
    torch.from_numpy(P_tr_phys).float(), torch.from_numpy(Q_tr_phys).float()), 
    batch_size=128, shuffle=True)

# ------------------------------------------
# 5. 极致基态训练 (300 轮 + 动态余弦退火)
# ------------------------------------------
model = PowerGridPINN(node_num=69).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)
loss_fn = PowerPINNLoss(G_tensor, B_tensor, obs_indices)

print("🔥 开始 300 轮双状态深度重构训练 (Proposed Method 5k有ARS版)...")
for epoch in range(300):
    lambda_p = 10 if epoch < 100 else 5000
    model.train()
    running_loss = 0.0
    for bx, by, bp, bq in train_loader:
        bx, by, bp, bq = bx.to(device), by.to(device), bp.to(device), bq.to(device)
        optimizer.zero_grad()
        
        mask_bx = apply_blind_zone(bx, obs_indices, PHYS_ZERO)
        vp, tp = model(mask_bx)
        
        # 传入真实的 V (by[:, :69]) 和 Theta (by[:, 69:]) 以供约束
        loss = loss_fn(vp, tp, bp, bq, by[:, :69], by[:, 69:], lambda_p, obs_weight=5e6)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    if epoch % 50 == 0 or epoch == 299:
        print(f"Epoch {epoch:<3} | Avg Loss: {running_loss/len(train_loader):.4e} | LR: {optimizer.param_groups[0]['lr']:.6e}")

# ------------------------------------------
# 6. 基态全量双状态统计审计
# ------------------------------------------
model.eval()
with torch.no_grad():
    tx = torch.from_numpy(X_te_norm).float().to(device)
    ty = torch.from_numpy(Y_te).float().to(device)
    t_mask = apply_blind_zone(tx, obs_indices, PHYS_ZERO)
        
    v_pred_all, t_pred_all = model(t_mask)
    v_true_all, t_true_all = ty[:, :69], ty[:, 69:]
    
    err_v_base = (v_pred_all - v_true_all).cpu().numpy()
    err_t_base = (t_pred_all - t_true_all).cpu().numpy()

mae_v_base = np.mean(np.abs(err_v_base))
rmse_v_base = np.sqrt(np.mean(err_v_base**2))
mae_t_base = np.mean(np.abs(err_t_base))
rmse_t_base = np.sqrt(np.mean(err_t_base**2))

print("\n" + "="*65)
print(f"🏆 [Base Case Validation] IEEE 69-Bus Proposed R-PINN (5k)")
print(f"🌍 Voltage (V) MAE  : {mae_v_base:.6e} p.u. | RMSE: {rmse_v_base:.6e} p.u.")
print(f"⚡ Phase Angle (T) MAE: {mae_t_base:.6e} deg  | RMSE: {rmse_t_base:.6e} deg")
print("="*65)

# ------------------------------------------
# 7. 辅助工具箱：SCI 级双子图升级版
# ------------------------------------------
def plot_academic_case_69_dual(v_true, v_pred, t_true, t_pred, mask, title, mae_v, rmse_v, mae_t, rmse_t, save_name):
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['axes.unicode_minus'] = False
    plt.style.use('seaborn-v0_8-paper')
    
    nodes = np.arange(69)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=600, sharex=True)
    
    dead_nodes = np.where(mask == 0)[0]
    
    # --- 图 (a): Voltage Magnitude ---
    if len(dead_nodes) > 0:
        ax1.fill_between(dead_nodes, -0.05, 1.15, color='gray', alpha=0.15, label='Topological Dead Zone')
        ax1.text(np.mean(dead_nodes), 0.5, 'ZERO-SHOT\nDEAD ZONE', color='darkred', 
                 fontsize=14, fontweight='bold', ha='center', va='center')
    else:
        ax1.axvspan(0, 68, color='gray', alpha=0.05, label='Blind Zone (No Sensor)')

    ax1.plot(nodes, v_true, 'k-', label='Ground Truth (Physics)', linewidth=2.0, zorder=3)
    ax1.plot(nodes, v_pred, 'r--', label='Proposed R-PINN', linewidth=1.5, zorder=4)
    
    live_obs = [idx for idx in obs_indices if mask[idx] == 1]
    if len(live_obs) > 0:
        ax1.scatter(live_obs, v_true[live_obs], color='blue', marker='*', s=200, label='PMU Sensors (15% Obs)', zorder=5)

    ax1.set_title(f"{title}: (a) Voltage Magnitude\n(MAE: {mae_v:.4e} p.u. | RMSE: {rmse_v:.4e} p.u.)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Voltage (p.u.)", fontsize=12)
    ax1.set_ylim(-0.05, 1.15)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)

    # --- 图 (b): Phase Angle ---
    if len(dead_nodes) > 0:
        ymin, ymax = min(t_true), max(t_true)
        y_margin = (ymax - ymin) * 0.1 if ymax != ymin else 1.0
        ax2.fill_between(dead_nodes, ymin - y_margin, ymax + y_margin, color='gray', alpha=0.15)
    else:
        ax2.axvspan(0, 68, color='gray', alpha=0.05)

    ax2.plot(nodes, t_true, 'k-', linewidth=2.0, zorder=3)
    ax2.plot(nodes, t_pred, 'b--', label='Proposed R-PINN (Theta)', linewidth=1.5, zorder=4)
    
    if len(live_obs) > 0:
        ax2.scatter(live_obs, t_true[live_obs], color='blue', marker='*', s=200, zorder=5)

    ax2.set_title(f"(b) Phase Angle\n(MAE: {mae_t:.4e} deg | RMSE: {rmse_t:.4e} deg)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Bus Index", fontsize=12)
    ax2.set_ylabel("Phase Angle (degree)", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight', dpi=600)
    # ⚠️ 核心修复：强制显示图片，并且移除 plt.close()
    plt.show()

# ------------------------------------------
# 8. 🌪️ Zero-Shot (零样本) N-1 双状态降维打击推演
# ------------------------------------------
print("\n" + "⚔️" * 15 + " 启动 N-1 Zero-shot 突发对账 " + "⚔️" * 15)

test_cases_n1 = [
    {"n": "C1 (Cut 9-10)", "csv": "data_69_C1.csv"},
    {"n": "C2 (Cut 35-36)", "csv": "data_69_C2.csv"},
    {"n": "C3 (Cut 54-55)", "csv": "data_69_C3.csv"}
]

model.eval()
with torch.no_grad():
    for case in test_cases_n1:
        try:
            df_c = pd.read_csv(case["csv"]).fillna(0.0)
            raw_c = df_c.values 
            
            # ⚠️ 核心切片：[P, Q, V, Theta] 提取前 69 节点
            P_c = raw_c[:, 0::4][:, :69]
            Q_c = raw_c[:, 1::4][:, :69]
            V_c = raw_c[:, 2::4][:, :69]
            T_c = raw_c[:, 3::4][:, :69] 
            
            X_c_raw = np.concatenate([P_c, Q_c], axis=1)
            X_c = torch.tensor(scaler.transform(X_c_raw), dtype=torch.float32).to(device)
            V_t = torch.tensor(V_c, dtype=torch.float32).to(device)
            T_t = torch.tensor(T_c, dtype=torch.float32).to(device)
            
            # 拓扑盲区物理识别：利用真实数据中的 0 电压构建天然掩码
            V_t_np = V_t[0].cpu().numpy()
            mask_np = np.where(V_t_np < 1e-3, 0.0, 1.0)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).to(device)
            
            # 零样本推理！同时拿到 V 和 Theta
            V_raw, T_raw = model(apply_blind_zone(X_c, obs_indices, PHYS_ZERO))
            
            V_final = V_raw * mask_t # 死区物理隔离抹零
            T_final = T_raw * mask_t 
            
            err_v_c = (V_final - V_t * mask_t).cpu().numpy()
            mae_v_c = np.mean(np.abs(err_v_c))
            rmse_v_c = np.sqrt(np.mean(err_v_c**2))
            
            # 同步结算相角误差
            err_t_c = (T_final - T_t * mask_t).cpu().numpy()
            mae_t_c = np.mean(np.abs(err_t_c))
            rmse_t_c = np.sqrt(np.mean(err_t_c**2))
            
            print(f"\n🔬 诊断场景: {case['n']}")
            print(f"   • Zero-shot V MAE5k小样本 : {mae_v_c:.6e} p.u. | RMSE: {rmse_v_c:.6e} p.u.")
            print(f"   • Zero-shot T MAE5k小样本 : {mae_t_c:.6e} deg  | RMSE: {rmse_t_c:.6e} deg")
            dead = np.where(mask_np == 0)[0]
            print(f"   🚫 成功隔离 {len(dead)} 个物理死区节点")
            
            plot_title = f"Zero-shot Reconstruction Profile: {case['n']}"
            save_filename = f"ieee69_zeroshot_5k_with_ars_{case['n'][:2]}_Dual.png"
            
            # 调用全新的双轨画图工具！
            plot_academic_case_69_dual(
                v_true=(V_t[0] * mask_t).cpu().numpy(),
                v_pred=V_final[0].cpu().numpy(),
                t_true=(T_t[0] * mask_t).cpu().numpy(),
                t_pred=T_final[0].cpu().numpy(),
                mask=mask_np,
                title=plot_title,
                mae_v=mae_v_c, rmse_v=rmse_v_c,
                mae_t=mae_t_c, rmse_t=rmse_t_c,
                save_name=save_filename
            )
            print(f"   ✅ 高清双轨科研图已显示并保存: {save_filename}")
            
        except Exception as e:
            print(f"❌ 运行故障 ({case['n']}): {e}")

print("\n🎉 IEEE 69 节点双状态(V&Theta)完全对齐！5k小样本有ars！跑完收工，出图！")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================================
# ⚙️ 1. 全局学术排版设置 (Times New Roman + 600 DPI)
# ==========================================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.style.use('seaborn-v0_8-paper')

# 场景标签
labels = ['Base Case', 'C1 (Cut 9-10)', 'C2 (Cut 35-36)', 'C3 (Cut 54-55)']
x = np.arange(len(labels))

# ==========================================================
# 📊 2. 核心数据录入 (根据你的最新终端日志精准提取)
# ==========================================================
# Proposed R-PINN (Mean-Loss)
mae_proposed = [0.001906, 0.002520, 0.001022, 0.001021]

# Ablation (Sum-Loss)
mae_sumloss  = [0.000529, 0.089866, 0.048997, 0.031301]

# Vanilla PINN (No ARS, 绝对裸奔)
mae_vanilla  = [0.262883, 0.133652, 0.165543, 0.256108]

width = 0.25
fig, ax = plt.subplots(figsize=(11, 7))

# 绘制三方会战柱状图 (颜色按照学术规范，红-烂，橙-中，蓝-最强)
rects1 = ax.bar(x - width, mae_proposed, width, label='Proposed R-PINN (Mean-Loss + ARS)', color='#1f77b4', edgecolor='black', alpha=0.9)
rects2 = ax.bar(x, mae_sumloss, width, label='Ablation (Sum-Loss + ARS)', color='#ff7f0e', edgecolor='black', alpha=0.9)
rects3 = ax.bar(x + width, mae_vanilla, width, label='Vanilla PINN (No ARS)', color='#d62728', edgecolor='black', alpha=0.9)

# 🚀 核心改进：在柱子上方显示数值（四位小数，不带科学计数法）
def add_labels(ax, rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),  # 垂直向上偏移 5 points
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

add_labels(ax, rects1)
add_labels(ax, rects2)
add_labels(ax, rects3)

# 🚨 对数轴设置 (因为Vanilla误差高达0.26，而Proposed小到0.001，必须用对数轴)
ax.set_yscale('log')
ax.set_ylabel('Mean Absolute Error (p.u.)', fontsize=14, fontweight='bold')
ax.set_title('Zero-Shot Generalization on IEEE 69-Bus System', fontsize=16, fontweight='bold', pad=25)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# 设置Y轴范围，给最上方的文字标签留出空间
ax.set_ylim(1e-4, 1.5) 
ax.grid(axis='y', linestyle='--', alpha=0.6, which='both')

# 图例放到正下方，排成一行，显得专业大气
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=11, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig('ieee69_generalization_n1_comparison_fixed.png', dpi=600, bbox_inches='tight')
print("✅ 第 3 张神图 (69 节点 N-1 泛化对比) 已生成！")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================================
# ⚙️ 1. 全局学术排版设置 (Times New Roman + 600 DPI)
# ==========================================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.style.use('seaborn-v0_8-paper')

# 场景标签
labels = ['Base Case', 'C1 (Cut 9-10)', 'C2 (Cut 35-36)', 'C3 (Cut 54-55)']
x = np.arange(len(labels))

# ==========================================================
# 📊 2. 核心数据录入 (根据最新的 5k 终端日志精准提取)
# ==========================================================
# Proposed R-PINN (Mean-Loss + ARS, 5k samples)
mae_proposed_5k = [0.001894, 0.000552, 0.000866, 0.001000]

# Vanilla PINN (No ARS, 绝对裸奔, 5k samples)
mae_vanilla_5k  = [0.364666, 0.237190, 0.272247, 0.361746]

width = 0.35
fig, ax = plt.subplots(figsize=(10, 6.5))

# 绘制双柱状图
rects1 = ax.bar(x - width/2, mae_proposed_5k, width, label='Proposed R-PINN (5k Samples)', color='#1f77b4', edgecolor='black', alpha=0.9)
rects2 = ax.bar(x + width/2, mae_vanilla_5k, width, label='Vanilla PINN (No ARS, 5k Samples)', color='#d62728', edgecolor='black', alpha=0.9)

# 🚀 核心改进：在柱子上方显示数值（四位小数）
def add_labels(ax, rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),  # 垂直向上偏移 5 points
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

add_labels(ax, rects1)
add_labels(ax, rects2)

# 🚨 对数轴设置 (Vanilla误差高达0.36，而Proposed小到0.0005，相差几百倍)
ax.set_yscale('log')
ax.set_ylabel('Mean Absolute Error (p.u.)', fontsize=14, fontweight='bold')
ax.set_title('Data Efficiency: Few-Shot Generalization on IEEE 69-Bus', fontsize=16, fontweight='bold', pad=25)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

# 设置Y轴范围，给最上方的文字标签留出空间
ax.set_ylim(1e-4, 2.0) 
ax.grid(axis='y', linestyle='--', alpha=0.6, which='both')

# 图例放到正下方
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=11, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig('ieee69_efficiency_sample_scaling_fixed.png', dpi=600, bbox_inches='tight')
print("✅ 69节点 5k少样本 (数据效率) 神图生成完毕！")
plt.show()